In [ ]:
# Import necessary libraries
!pip install datasets -q
!pip install scikit-learn -q
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np
import time
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from collections import Counter

try:
    import subprocess
    import sys
    print("Installing necessary libraries")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "scikit-learn", "-q"])
    print("Libraries installed successfully.")
except Exception as e:
    print(f"Error installing libraries: {e}")
    exit()

torch.manual_seed(42)
np.random.seed(42)

#GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Installing necessary libraries
Libraries installed successfully.
Using device: cuda


In [ ]:
#This will install the 'datasets' and 'scikit-learn' libraries.
try:
    import subprocess
    import sys
    print("--- Installing necessary libraries ---")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "scikit-learn", "-q"])
    print("Libraries installed.")
except Exception as e:
    print(f"Couldn't install libraries. Error: {e}")
    exit()

# Alright, let's get our data ready for the model.
print("\nFetching the dataset... this might take a moment.")

data_is_ready = False
try:
    # We're forcing a fresh download here. It's a good trick to avoid
    # frustrating issues caused by a corrupted cache.
    full_dataset = load_dataset(
        "google/code_x_glue_cc_code_completion_token",
        'python',
        download_mode="force_redownload"
    )

    print(f"Success! Found these splits: {list(full_dataset.keys())}")

    # The dataset should have 'train' and 'test' splits. Let's make sure.
    if 'train' in full_dataset and 'test' in full_dataset:
        print("Looks like we have the 'train' and 'test' data we need.")

        # We'll use their 'train' data for our own training and validation sets.
        training_pool_df = full_dataset['train'].to_pandas()
        test_df = full_dataset['test'].to_pandas() # This is our final exam data.

        print(f"Loaded {len(training_pool_df)} examples for training/validation.")
        print(f"And {len(test_df)} examples set aside for the final test.")
        data_is_ready = True
    else:
        # If something is missing, we can't continue.
        print("Uh oh, the dataset seems to be missing the 'train' or 'test' split.")

except Exception as e:
    print(f"Something went wrong while downloading the data. Error: {e}")

# We only move on if the data was actually loaded.
if data_is_ready:
    # Now, let's carve out a validation set from our training data.
    print("\nSplitting the data into training (80%) and validation (20%)...")
    # Using random_state=42 is a classic move to make sure our split is the same every time.
    train_df, val_df = train_test_split(
        training_pool_df,
        test_size=0.2,
        random_state=42
    )
    print(f"Done. We have {len(train_df)} for training and {len(val_df)} for validation.")

    # A model only understands numbers, not words. So we need to build a vocabulary,
    # basically a dictionary of all the words our model should know.
    print("\nBuilding the vocabulary from the training data...")
    # It's important to only use the training data to build the vocab to avoid "cheating"
    # by peeking at the validation or test sets.
    all_tokens = [token for code_list in train_df['code'] for token in code_list]
    word_counts = Counter(all_tokens)

    # We can't use every single word, so we'll just take the 10,000 most common ones.
    top_tokens = word_counts.most_common(10000)
    vocab = [token for token, _ in top_tokens]

    # We also need to add a few special tokens for handling things like padding and rare words.
    special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>']
    vocab = special_tokens + vocab
    print(f"Vocabulary created with a total of {len(vocab)} tokens.")

    # Last step: create our lookup tables to turn tokens into numbers and back again.
    print("\nCreating the token-to-index mappings...")
    token_to_idx = {token: idx for idx, token in enumerate(vocab)}
    idx_to_token = {idx: token for idx, token in enumerate(vocab)}

    # These will be super useful later.
    UNK_IDX = token_to_idx['<UNK>']
    PAD_IDX = token_to_idx['<PAD>']
    print(f"The padding token <PAD> is index: {PAD_IDX}")
    print(f"The unknown token <UNK> is index: {UNK_IDX}")

--- Installing necessary libraries ---
Libraries installed.

Fetching the dataset... this might take a moment.


python/train-00000-of-00002.parquet:   0%|          | 0.00/70.1M [00:00<?, ?B/s]

python/train-00001-of-00002.parquet:   0%|          | 0.00/70.9M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/69.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Success! Found these splits: ['train', 'test']
Looks like we have the 'train' and 'test' data we need.
Loaded 100000 examples for training/validation.
And 50000 examples set aside for the final test.

Splitting the data into training (80%) and validation (20%)...
Done. We have 80000 for training and 20000 for validation.

Building the vocabulary from the training data...
Vocabulary created with a total of 10004 tokens.

Creating the token-to-index mappings...
The padding token <PAD> is index: 0
The unknown token <UNK> is index: 1


In [ ]:
# --- Part 1: The Recipe for a Single Piece of Data ---

# We need a way to tell PyTorch how to handle our data. A custom Dataset
# class is the standard way to do this. Think of it as a recipe for
# preparing one item from our dataframe.
class CodeDataset(Dataset):
    # This just sets up our dataset with the dataframe and our token map.
    def __init__(self, df, token_to_idx, max_len=1000):
        self.df = df
        self.token_to_idx = token_to_idx
        self.max_len = max_len
        self.pad_idx = token_to_idx['<PAD>']
        self.unk_idx = token_to_idx['<UNK>']

    # A simple function to tell PyTorch how many items are in our dataset.
    def __len__(self):
        return len(self.df)

    # This is the core magic. It defines what happens when we ask for an item.
    def __getitem__(self, index):
        # 1. Grab a line of code (which is already a list of tokens).
        tokens = self.df.iloc[index]['code']

        # 2. Chop it down if it's too long for our model.
        # We cap it at max_len - 1 to make room for the target.
        if len(tokens) > self.max_len - 1:
            tokens = tokens[:self.max_len - 1]

        # 3. Turn the text tokens into numbers our model can actually use.
        # If a word isn't in our vocab, we just use the <UNK> token's number.
        numbered_tokens = [self.token_to_idx.get(t, self.unk_idx) for t in tokens]

        # 4. Create the input and the target. For a language model, the target is
        # just the input sequence shifted over by one.
        # So if the model sees "import torch as", we want it to predict "torch as nn".
        input_sequence = numbered_tokens
        target_sequence = numbered_tokens[1:]

        # 5. Pad everything to a fixed length (1000 tokens in our case).
        # Models need fixed-size inputs, so we fill the rest with our <PAD> token's number.
        input_padding = [self.pad_idx] * (self.max_len - len(input_sequence))
        target_padding = [self.pad_idx] * (self.max_len - len(target_sequence))

        x = torch.tensor(input_sequence + input_padding, dtype=torch.long)
        y = torch.tensor(target_sequence + target_padding, dtype=torch.long)

        return x, y


# --- Part 2: Putting it all together with DataLoaders ---

# Let's set some constants for our data pipeline.
# Batch size of 32 is a pretty standard place to start.
BATCH_SIZE = 32
MAX_SEQ_LEN = 1000

print("Setting up the DataLoaders...")

# Now we use our "recipe" (CodeDataset) to create datasets for training and validation.
train_dataset = CodeDataset(train_df, token_to_idx, max_len=MAX_SEQ_LEN)
val_dataset = CodeDataset(val_df, token_to_idx, max_len=MAX_SEQ_LEN)

# The DataLoader is a handy PyTorch tool that takes our dataset and serves up
# shuffled batches of data, ready for training.
# Shuffling the training data is super important – it helps the model learn better.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("✅ DataLoaders are ready to go!")
print(f"We have {len(train_loader)} batches for training each epoch.")
print(f"And {len(val_loader)} batches for validation each epoch.")


# --- Part 3: A quick sanity check ---

# I always like to pull out one batch to eyeball it and make sure the shapes look right.
print("\n--- Grabbing a sample batch to check the shapes ---")
x_batch, y_batch = next(iter(train_loader))

# The shape should be [batch_size, sequence_length]
print(f"Input batch shape looks good: {x_batch.shape}")
print(f"Target batch shape also looks good: {y_batch.shape}")

Setting up the DataLoaders...
✅ DataLoaders are ready to go!
We have 2500 batches for training each epoch.
And 625 batches for validation each epoch.

--- Grabbing a sample batch to check the shapes ---
Input batch shape looks good: torch.Size([32, 1000])
Target batch shape also looks good: torch.Size([32, 1000])


In [ ]:
# --- Part 1: Defining the Model's Blueprint ---

# Okay, time to build our first model, the simple RNN.
# We'll define it as a class, which is the standard PyTorch way.
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, dropout_rate, pad_token_id):
        super().__init__()

        # 1. The Embedding Layer
        # This is a crucial first step. It's basically a big lookup table that turns
        # our word IDs (like 52) into dense vectors (like a list of 256 numbers).
        # The model learns what these vectors should be.
        # We also tell it to ignore the padding token, which is important for training.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)

        # 2. The RNN Core
        # This is the heart of the model, where the sequence processing happens.
        # 'batch_first=True' is a lifesaver - it just means our data tensors
        # will be shaped like [batch, sequence, features], which is more intuitive.
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout_rate,  # This adds dropout between RNN layers (if num_layers > 1)
            batch_first=True
        )

        # 3. Another Dropout Layer
        # We'll apply this after the RNN and before the final prediction layer
        # to help prevent the model from just memorizing the training data.
        self.dropout = nn.Dropout(dropout_rate)

        # 4. The Final Prediction Layer
        # This is a standard linear layer that takes the RNN's output and maps it
        # to a score for every single word in our vocabulary. The highest score wins!
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    # The forward pass defines how data flows through the layers we just defined.
    def forward(self, text_batch):
        # text_batch starts as -> [batch_size, sequence_length]

        # 1. Turn word IDs into vectors.
        embedded_text = self.embedding(text_batch)
        # embedded_text is now -> [batch_size, sequence_length, embedding_dim]

        # 2. Process the sequence with the RNN.
        # We mostly care about the `rnn_output` here, which contains the hidden state
        # for every single token in the sequence.
        rnn_output, last_hidden_state = self.rnn(embedded_text)
        # rnn_output is now -> [batch_size, sequence_length, hidden_size]

        # 3. Apply dropout for regularization.
        output_after_dropout = self.dropout(rnn_output)

        # 4. Get the final scores for each word in the vocab.
        predictions = self.output_layer(output_after_dropout)
        # predictions are now -> [batch_size, sequence_length, vocab_size]

        return predictions


# --- Part 2: Actually Building the Model ---

print("Setting up the model with our hyperparameters...")
# These are the specs from the assignment.
# It's good practice to keep them in one place.
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT_RATE = 0.5  # 50% is a pretty aggressive but common starting point.

# Let's create an instance of our model.
rnn_model = SimpleRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE,
    pad_token_id=PAD_IDX
)

# --- Part 3: Checking Our Work ---

# It's always a good idea to see how big the model is.
def count_params(model):
    # This just counts up all the numbers in the model that can be trained.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nCool, the RNN model is built. It has {count_params(rnn_model):,} trainable parameters.")
print("\nHere's a look at the architecture:")
print(rnn_model)

# The final sanity check: let's push a batch of data through it and see if the output shape is what we expect.
# We use 'torch.no_grad()' because we're just checking, not training, so we don't need to compute gradients.
with torch.no_grad():
    # Let's assume x_batch is still in memory from the previous step
    output = rnn_model(x_batch)

print("\n--- Final Shape Verification ---")
print(f"The model's output shape is: {output.shape}")
print(f"Which matches our expected shape of [batch_size, seq_len, vocab_size]:")
print(f"[{BATCH_SIZE}, {MAX_SEQ_LEN}, {VOCAB_SIZE}]")
print("Looks like everything is wired up correctly!")

Setting up the model with our hyperparameters...

Cool, the RNN model is built. It has 8,612,628 trainable parameters.

Here's a look at the architecture:
SimpleRNN(
  (embedding): Embedding(10004, 256, padding_idx=0)
  (rnn): RNN(256, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (output_layer): Linear(in_features=512, out_features=10004, bias=True)
)

--- Final Shape Verification ---
The model's output shape is: torch.Size([32, 1000, 10004])
Which matches our expected shape of [batch_size, seq_len, vocab_size]:
[32, 1000, 10004]
Looks like everything is wired up correctly!


In [ ]:
# --- Setting up the training rig ---

# First, let's see if we have a GPU to work with. Training on a CPU would take forever.
import math
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Alright, we're running on: {device}")

# Gotta move our model over to the GPU (or CPU if that's all we have).
# Let's assume 'rnn_model' is our model from the last step.
rnn_model = rnn_model.to(device)

# We'll use the Adam optimizer. It's a solid, all-around choice that usually works well.
optimizer = optim.Adam(rnn_model.parameters())

# For the loss function, CrossEntropyLoss is the standard for this kind of task.
# The `ignore_index` part is SUPER important. It tells PyTorch to completely ignore
# the <PAD> tokens when calculating the loss, so the model isn't punished for
# its predictions on padding we added ourselves.
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


# --- Defining our game plan for one epoch ---

# This function will handle one full pass over the training data.
def run_train_epoch(model, data_loader, optimizer, loss_fn):
    # Flip the model into 'train' mode. This tells layers like Dropout to be active.
    model.train()
    total_loss = 0

    # Loop through all the batches from our data loader.
    for inputs, targets in data_loader:
        # Move the data for this batch to the GPU.
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Always gotta clear out the gradients from the last step.
        optimizer.zero_grad()

        # Get the model's predictions.
        predictions = model(inputs)

        # The loss function expects a 2D list of predictions and a 1D list of targets,
        # so we have to flatten our batches out.
        loss = loss_fn(predictions.view(-1, VOCAB_SIZE), targets.view(-1))

        # This is where the magic happens: PyTorch calculates all the gradients.
        loss.backward()

        # A common problem with RNNs is "exploding gradients". This next line is a safety
        # rail that clips the gradients to a manageable size to keep training stable.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        # Now, tell the optimizer to actually update the model's weights based on the gradients.
        optimizer.step()

        total_loss += loss.item()

    # Return the average loss for the epoch.
    return total_loss / len(data_loader)

# This function will check how our model is doing on the validation set.
def run_eval_epoch(model, data_loader, loss_fn):
    # Switch to 'eval' mode. This turns off things like Dropout for consistent predictions.
    model.eval()
    total_loss = 0

    # We wrap this in `torch.no_grad()` because we're just evaluating, not learning.
    # This tells PyTorch not to bother calculating gradients, which saves a lot of memory and speed.
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            predictions = model(inputs)
            loss = loss_fn(predictions.view(-1, VOCAB_SIZE), targets.view(-1))
            total_loss += loss.item()

    return total_loss / len(data_loader)


# --- Let's kick off the training! ---

# Let's just do 1 epoch for now to see how it goes.
NUM_EPOCHS = 1
best_so_far_loss = float('inf') # Start with an infinitely bad loss.

print("\n--- Starting the RNN training loop! ---")

for epoch in range(NUM_EPOCHS):
    print(f"\nStarting Epoch {epoch + 1}...")
    start_time = time.time()

    # Run a full training epoch, then a full validation epoch.
    train_loss = run_train_epoch(rnn_model, train_loader, optimizer, criterion)
    valid_loss = run_eval_epoch(rnn_model, val_loader, criterion)

    end_time = time.time()
    mins, secs = divmod(end_time - start_time, 60)

    # If this is the best model we've seen so far, let's save its weights.
    if valid_loss < best_so_far_loss:
        best_so_far_loss = valid_loss
        torch.save(rnn_model.state_dict(), 'best-rnn-model.pt')
        print(f"\tNew best validation loss: {valid_loss:.3f}. Model saved!")

    print(f'Finished Epoch {epoch + 1} in {int(mins)}m {int(secs)}s')
    # Perplexity (PPL) is just another way to think about the loss. Lower is better.
    print(f'\tTraining Loss: {train_loss:.3f} | Training PPL: {math.exp(train_loss):.2f}')
    print(f'\tValidation Loss: {valid_loss:.3f} | Validation PPL: {math.exp(valid_loss):.2f}')

print("\n--- All done! ---")
print(f"The best validation loss we achieved was {best_so_far_loss:.3f}")

Alright, we're running on: cuda

--- Starting the RNN training loop! ---

Starting Epoch 1...
	New best validation loss: 1.956. Model saved!
Finished Epoch 1 in 16m 58s
	Training Loss: 2.161 | Training PPL: 8.68
	Validation Loss: 1.956 | Validation PPL: 7.07

--- All done! ---
The best validation loss we achieved was 1.956


In [ ]:
# --- Part 1: Setting the Stage ---
# This part is a bit of a safety net. If the notebook has been restarted,
# we need to redefine our model's blueprint and all its settings before we
# can load our saved weights.

# First, let's lock in the model's hyperparameters again.
EMBED_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

# Now, let's make sure our data-related variables are still loaded.
try:
    VOCAB_SIZE = len(vocab)
    PAD_IDX = token_to_idx['<PAD>']
except NameError:
    print("Whoops! Looks like the 'vocab' or 'token_to_idx' variables are missing.")
    print("Please make sure to re-run the Task 1 (Data Preparation) cell first, then try this again.")
    raise

# We also need the model's class definition and our helper functions in memory.
class RNNModel(nn.Module):
    # This is the blueprint for our simple RNN.
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        # Changed 'fc_out' to 'output_layer' to match the saving model
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, text):
        embedded = self.embedding(text)
        rnn_output, _ = self.rnn(embedded)
        # Changed 'fc_out' to 'output_layer' to match the saving model
        predictions = self.output_layer(self.dropout(rnn_output))
        return predictions

# Helper function (Keeping the name 'get_model_loss' for consistency, though 'run_eval_epoch' is used elsewhere)
def get_model_loss(model, data_loader, loss_fn):
    # A quick function to check the model's loss on a given dataset.
    model.eval()
    total_loss = 0
    with torch.no_grad(): # Super important: tells PyTorch we're not training, saving time and memory.
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = loss_fn(preds.view(-1, VOCAB_SIZE), y.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)

# Helper function (Keeping the name 'check_top5_accuracy' for consistency)
def check_top5_accuracy(model, data_loader, device, pad_token_id):
    # This checks how often the right answer is in the model's top 5 guesses.
    model.eval()
    total_correct, total_tokens = 0, 0
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            # This is the cool part: ask PyTorch for the top 5 predictions.
            _, top_k = torch.topk(preds, 5, dim=2)
            # Check if the real answer (y) is anywhere in our top_k guesses.
            correct = (top_k == y.unsqueeze(-1)).any(dim=2)
            # We don't want to count padding tokens as "correct" or "incorrect", so we mask them out.
            non_pad_mask = (y != pad_token_id)
            total_correct += (correct & non_pad_mask).sum().item()
            total_tokens += non_pad_mask.sum().item()
    return (total_correct / total_tokens) * 100

# --- Part 2: The Main Event - Loading and Evaluating ---

print("\n--- Time to evaluate our trained RNN! ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on device: {device}")

# First, we build an empty "skeleton" of the model.
rnn_model = RNNModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, PAD_IDX).to(device)
print("Model structure has been re-created.")

print("Attempting to load the saved weights...")
try:
    # Now, we load our saved weights into the skeleton.
    rnn_model.load_state_dict(torch.load('best-rnn-model.pt'))
    print("Success! Loaded the best checkpoint from training.")

    # We need the loss function again to calculate perplexity.
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    # Let's run our checks on the validation data.
    final_valid_loss = get_model_loss(rnn_model, val_loader, criterion)
    final_top5_acc = check_top5_accuracy(rnn_model, val_loader, device, PAD_IDX)

    print("\n--- RNN Final Scorecard (Validation Set) ---")
    print(f"Perplexity: {math.exp(final_valid_loss):.3f}")
    print(f"Top-5 Accuracy: {final_top5_acc:.2f}%")

except FileNotFoundError:
    print("\nERROR: Couldn't find the 'best-rnn-model.pt' file.")
    print("Make sure you've successfully run the training loop at least once to create it!")


--- Time to evaluate our trained RNN! ---
Running on device: cuda
Model structure has been re-created.
Attempting to load the saved weights...
Success! Loaded the best checkpoint from training.

--- RNN Final Scorecard (Validation Set) ---
Perplexity: 7.071
Top-5 Accuracy: 81.89%


In [ ]:
#Part 1: Building the "Smarter" LSTM Model

# Time for the second model. The LSTM is like the smarter, more sophisticated cousin of the simple RNN.
# It's generally much better at remembering things from way back in a sequence.
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, dropout_rate, pad_token_id):
        super().__init__()

        # The embedding layer is the same as before. It's our trusty word-to-vector lookup table.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)

        # Here's the key change: we're swapping out nn.RNN for nn.LSTM.
        # Under the hood, the LSTM has a more complex setup with special "gates"
        # that allow it to intelligently decide what information to remember and what to forget.
        # This is a huge advantage for understanding code, where a variable might be
        # defined many lines before it's used.
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            dropout=dropout_rate, # This dropout is applied between LSTM layers.
            batch_first=True    # Still using this for a more intuitive tensor shape.
        )

        # We'll use dropout again after the LSTM for regularization.
        self.dropout = nn.Dropout(dropout_rate)

        # And the final linear layer to get our vocabulary scores is the same as before.
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    def forward(self, text_batch):
        # The data flow is pretty much the same as the RNN.
        # 1. Get the word vectors.
        embedded_text = self.embedding(text_batch)

        # 2. Process the sequence through the LSTM.
        # The LSTM actually returns three things: the output for every token, the final
        # hidden state, and the final "cell state" (its long-term memory).
        # For our goal of predicting the next word at each step, we only need the main `lstm_output`.
        lstm_output, (last_hidden, last_cell) = self.lstm(embedded_text)

        # 3. Apply dropout and get the final predictions.
        output_after_dropout = self.dropout(lstm_output)
        predictions = self.output_layer(output_after_dropout)

        return predictions


# --- Part 2: Instantiating the LSTM ---

# We'll use the same settings as the RNN to make sure it's a fair comparison.
# Let's assume the hyperparameter variables (VOCAB_SIZE, etc.) are still in memory.
lstm_model = LSTMModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT,
    pad_token_id=PAD_IDX
)


# --- Part 3: Comparing the two models ---

# A little helper to count the trainable weights in a model.
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("--- Model Comparison ---")
# Let's assume 'rnn_model' still exists from the previous step for this comparison.
print(f"The simple RNN model had {count_params(rnn_model):,} parameters.")
print(f"Our new LSTM model has {count_params(lstm_model):,} parameters.")
print("\nThat's a pretty big jump! The extra parameters come from all the internal gates (forget, input, output)")
print("that make the LSTM better at handling memory. It's a classic trade-off: more power for more complexity.")

print("\nLet's take a look at the LSTM's architecture:")
print(lstm_model)

--- Model Comparison ---
The simple RNN model had 8,612,628 parameters.
Our new LSTM model has 11,371,284 parameters.

That's a pretty big jump! The extra parameters come from all the internal gates (forget, input, output)
that make the LSTM better at handling memory. It's a classic trade-off: more power for more complexity.

Let's take a look at the LSTM's architecture:
LSTMModel(
  (embedding): Embedding(10004, 256, padding_idx=0)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (output_layer): Linear(in_features=512, out_features=10004, bias=True)
)


In [ ]:
#Time to train our new LSTM model

# First, let's get the setup right. We want to be on the GPU if possible.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"We're training on the: {device}")

# Move the LSTM model we just created onto the right hardware.
lstm_model = lstm_model.to(device)

# We need a new optimizer for this new model. We can't reuse the old one,
# as it's tied to the RNN's parameters. Adam is a good default.
optimizer = optim.Adam(lstm_model.parameters())

# We can totally reuse our loss function ('criterion') and our training/evaluation
# functions ('train_one_epoch' and 'evaluate') from before. No need to rewrite them!
# This is why writing functions is so handy.

# --- Kicking off the training loop for the LSTM ---

# Let's stick to the same number of epochs as the RNN to keep the comparison fair.
NUM_EPOCHS = 1
# We'll track the best validation loss for this model separately.
best_lstm_loss = float('inf') # Start at infinity

print("\n--- Starting the LSTM training! ---")
print("Heads up: this will likely take a bit longer than the simple RNN did.")
print("All those extra gates in the LSTM add up to more number crunching.")

for epoch in range(NUM_EPOCHS):
    start_time = time.time() # Start the timer for the epoch

    # The game plan is exactly the same as before.
    # One pass over the training data to learn...
    train_loss = run_train_epoch(lstm_model, train_loader, optimizer, criterion)
    # ...and one pass over the validation data to see how well we're doing.
    valid_loss = run_eval_epoch(lstm_model, val_loader, criterion)

    end_time = time.time() # Stop the timer
    mins, secs = divmod(end_time - start_time, 60)

    # Check if this is the best version of the model we've seen yet.
    if valid_loss < best_lstm_loss:
        best_lstm_loss = valid_loss
        # If it is, save its weights to a new file. We don't want to overwrite our best RNN model.
        torch.save(lstm_model.state_dict(), 'best-lstm-model.pt')
        print(f"\tNew best validation loss: {valid_loss:.3f}. Saving model checkpoint!")

    print(f'\nFinished Epoch {epoch + 1} in {int(mins)}m {int(secs)}s')
    # Let's print the stats. Perplexity is just a more intuitive way to look at the loss.
    print(f'\tTraining Loss: {train_loss:.3f} | Training PPL: {math.exp(train_loss):.2f}')
    print(f'\tValidation Loss: {valid_loss:.3f} | Validation PPL: {math.exp(valid_loss):.2f}')


print("\n--- And we're done with the LSTM training! ---")
print(f"The best validation loss this model achieved was {best_lstm_loss:.3f}")

We're training on the: cuda

--- Starting the LSTM training! ---
Heads up: this will likely take a bit longer than the simple RNN did.
All those extra gates in the LSTM add up to more number crunching.
	New best validation loss: 1.941. Saving model checkpoint!

Finished Epoch 1 in 25m 49s
	Training Loss: 2.534 | Training PPL: 12.60
	Validation Loss: 1.941 | Validation PPL: 6.97

--- And we're done with the LSTM training! ---
The best validation loss this model achieved was 1.941


In [ ]:
# --- Step 1: Load the best saved LSTM model ---
# First, ensure the LSTM model structure is defined and instantiated.
# This prevents errors if your notebook kernel has restarted.
print("Re-creating the LSTM model structure...")
lstm_model = LSTMModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBED_DIM,
    hidden_size=HIDDEN_DIM,
    num_layers=N_LAYERS,
    dropout_rate=DROPOUT,
    pad_token_id=PAD_IDX
).to(device)

# Now, load the weights from the file
try:
    lstm_model.load_state_dict(torch.load('best-lstm-model.pt'))
    print("Loaded the best LSTM model checkpoint for evaluation.")

    # --- Step 2: Calculate its Top-5 Accuracy ---
    lstm_val_top5_accuracy = check_top5_accuracy(lstm_model, val_loader, device, PAD_IDX)

    print("\n--- LSTM Model Validation Performance ---")
    print(f"Best Validation Perplexity: {math.exp(best_lstm_loss):.3f}")
    print(f"Validation Top-5 Accuracy: {lstm_val_top5_accuracy:.2f}%")

except FileNotFoundError:
    print("ERROR: 'best-lstm-model.pt' not found. Please re-run the LSTM training cell to create the file.")
except NameError as e:
    print(f"ERROR: A variable is not defined: {e}. Please re-run the Task 1 cell and the LSTM model definition cell.")

Re-creating the LSTM model structure...


NameError: name 'LSTMModel' is not defined

In [ ]:
# --- Part 1: Building the souped-up Residual LSTM ---

# Okay, for our final trick (Task 3), we're going to take our best model (the LSTM)
# and give it a 'residual connection'.
# The whole idea is to create a shortcut for information to travel across layers.
# Instead of just going from Layer 1 -> Layer 2, we're going to make the final output
# a mix of both: Output = Layer2_Output + Layer1_Output.
# This often helps the model train better and is a super common technique in modern deep learning.
class ResidualLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, dropout_rate, pad_token_id):
        super().__init__()

        # The first and last layers are the same as our baseline LSTM.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(hidden_size, vocab_size)

        # Now for the tricky part. To get that 'shortcut' connection, we can't just
        # stack the LSTMs like we did before (with num_layers=2). We need to create them
        # as two separate, individual layers so we can grab the output from each one.
        self.lstm_layer1 = nn.LSTM(embedding_dim, hidden_size, num_layers=1, batch_first=True)
        self.lstm_layer2 = nn.LSTM(hidden_size, hidden_size, num_layers=1, batch_first=True)

    def forward(self, text_batch):
        # 1. Start with the embeddings, as usual.
        embedded_text = self.embedding(text_batch)

        # 2. Get the output from the first LSTM layer. Let's call it H1.
        H1, _ = self.lstm_layer1(embedded_text)

        # 3. Now, feed H1 into the second LSTM layer to get H2.
        H2, _ = self.lstm_layer2(H1)

        # 4. And here's the magic trick! This is the residual connection.
        # We just add the output of the first layer to the output of the second.
        # This gives the network a "shortcut" for information to flow.
        H_final = H1 + H2

        # 5. Finally, apply dropout and pass it through the output layer for our predictions.
        output_after_dropout = self.dropout(H_final)
        predictions = self.output_layer(output_after_dropout)

        return predictions


# --- Part 2: Putting it all together ---

# Let's create an instance of our new model. The settings are the same.
# We'll assume the hyperparameter variables (VOCAB_SIZE, etc.) are still in memory from previous cells.
residual_lstm_model = ResidualLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBED_DIM,
    hidden_size=HIDDEN_DIM,
    num_layers=N_LAYERS, # Conceptually it's still 2 layers, we just built it differently.
    dropout_rate=DROPOUT,
    pad_token_id=PAD_IDX
)

# A quick sanity check on the parameter count.
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("--- Comparing Model Sizes ---")
# Let's assume 'lstm_model' from the previous step is still around for this comparison.
print(f"Our baseline LSTM had {count_params(lstm_model):,} parameters.")
print(f"Our new Residual LSTM has {count_params(residual_lstm_model):,} parameters.")
print("\nInteresting! They're exactly the same. The residual connection is just a change in how")
print("we wire things up; it doesn't actually add any new weights for the model to learn.")

print("\nHere's what the new architecture looks like:")
print(residual_lstm_model)

The baseline LSTM model has 11,371,284 trainable parameters.
The Residual LSTM model has 11,371,284 trainable parameters.

--- Residual LSTM Model Architecture ---
ResidualLSTMModel(
  (embedding): Embedding(10004, 256, padding_idx=0)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc_out): Linear(in_features=512, out_features=10004, bias=True)
  (lstm1): LSTM(256, 512, batch_first=True)
  (lstm2): LSTM(512, 512, batch_first=True)
)


In [ ]:
# --- Step 1: Setup for the new model ---

# Ensure the model is on the correct device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
residual_lstm_model = residual_lstm_model.to(device)
print(f"Using device: {device}")

# Create a new optimizer for this specific model
optimizer = optim.Adam(residual_lstm_model.parameters())

# We will use the same criterion (CrossEntropyLoss with ignore_index=PAD_IDX) as before.

# --- Step 2: Training Loop for the Residual LSTM Model ---

N_EPOCHS = 1
best_residual_valid_loss = float('inf')

print("\n--- Starting Residual LSTM Model Training ---")
print("Training time should be very similar to the baseline LSTM.")

for epoch in range(N_EPOCHS):

    start_time = time.time()

    # Use the same training and evaluation functions
    train_loss = train_one_epoch(residual_lstm_model, train_loader, optimizer, criterion)
    valid_loss = evaluate(residual_lstm_model, val_loader, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    # Save the best model to a new file
    if valid_loss < best_residual_valid_loss:
        best_residual_valid_loss = valid_loss
        torch.save(residual_lstm_model.state_dict(), 'best-residual-lstm-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

print("\n--- Residual LSTM Training Finished ---")
print(f"The best validation loss achieved by the Residual LSTM model was {best_residual_valid_loss:.3f}")

Using device: cuda

--- Starting Residual LSTM Model Training ---
Training time should be very similar to the baseline LSTM.
Epoch: 01 | Time: 28m 19s
	Train Loss: 1.598 | Train PPL:   4.943
	 Val. Loss: 1.462 |  Val. PPL:   4.313

--- Residual LSTM Training Finished ---
The best validation loss achieved by the Residual LSTM model was 1.462


In [ ]:
# --- Step 1: Load the best saved Residual LSTM model ---
# Re-define the model structure to prevent any errors.
print("Re-creating the Residual LSTM model structure...")
residual_lstm_model = ResidualLSTMModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(device)

# Now, load the weights from the file
try:
    residual_lstm_model.load_state_dict(torch.load('best-residual-lstm-model.pt'))
    print("Loaded the best Residual LSTM model checkpoint for evaluation.")

    # --- Step 2: Calculate its Top-5 Accuracy ---
    residual_lstm_val_top5_accuracy = calculate_top5_accuracy(residual_lstm_model, val_loader, device, PAD_IDX)

    print("\n--- Residual LSTM Model Validation Performance ---")
    print(f"Best Validation Perplexity: {math.exp(best_residual_valid_loss):.3f}")
    print(f"Validation Top-5 Accuracy: {residual_lstm_val_top5_accuracy:.2f}%")

except FileNotFoundError:
    print("ERROR: 'best-residual-lstm-model.pt' not found. Please re-run the Residual LSTM training cell to create the file.")
except NameError as e:
    print(f"ERROR: A variable is not defined: {e}. Please re-run the Task 1 cell and the model definition cell.")

Re-creating the Residual LSTM model structure...
Loaded the best Residual LSTM model checkpoint for evaluation.

--- Residual LSTM Model Validation Performance ---
Best Validation Perplexity: 4.313
Validation Top-5 Accuracy: 87.45%


In [ ]:
# --- Step 1: Create the Test DataLoader ---
# This is done only once, at the very end.
test_dataset = CodeDataset(test_df, token_to_idx, max_seq_len=MAX_SEQ_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Test DataLoader created with {len(test_loader)} batches.")

# --- Step 2: Evaluate all three best models on the test set ---

# Ensure all models and functions are defined (in case of kernel restart)
# If you run into a NameError here, re-run the cells where the models were defined.

# Move models to the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rnn_model.to(device)
lstm_model.to(device)
residual_lstm_model.to(device)

print("\n--- Evaluating FINAL models on the TEST SET ---")

# --- RNN Final Evaluation ---
rnn_model.load_state_dict(torch.load('best-rnn-model.pt'))
rnn_test_loss = evaluate(rnn_model, test_loader, criterion)
rnn_test_ppl = math.exp(rnn_test_loss)
rnn_test_top5_acc = calculate_top5_accuracy(rnn_model, test_loader, device, PAD_IDX)
print("RNN Evaluation Complete.")

# --- Baseline LSTM Final Evaluation ---
lstm_model.load_state_dict(torch.load('best-lstm-model.pt'))
lstm_test_loss = evaluate(lstm_model, test_loader, criterion)
lstm_test_ppl = math.exp(lstm_test_loss)
lstm_test_top5_acc = calculate_top5_accuracy(lstm_model, test_loader, device, PAD_IDX)
print("Baseline LSTM Evaluation Complete.")

# --- Residual LSTM Final Evaluation ---
residual_lstm_model.load_state_dict(torch.load('best-residual-lstm-model.pt'))
residual_lstm_test_loss = evaluate(residual_lstm_model, test_loader, criterion)
residual_lstm_test_ppl = math.exp(residual_lstm_test_loss)
residual_lstm_test_top5_acc = calculate_top5_accuracy(residual_lstm_model, test_loader, device, PAD_IDX)
print("Residual LSTM Evaluation Complete.")

Test DataLoader created with 1563 batches.

--- Evaluating FINAL models on the TEST SET ---
RNN Evaluation Complete.
Baseline LSTM Evaluation Complete.
Residual LSTM Evaluation Complete.


In [ ]:
# --- This code prints the final report based on the variables from the last step ---

print("="*60)
print("              TASK 4: FINAL REPORT & SUBMISSION")
print("="*60)


# --- 1. Results: Baseline Comparison ---
print("\n### 1. Results: Baseline Comparison (on Test Set)\n")
print(f"{'Metric':<25} | {'RNN Model':<15} | {'LSTM Model':<15}")
print("-"*60)
print(f"{'Test Perplexity (PPL)':<25} | {rnn_test_ppl:<15.3f} | {lstm_test_ppl:<15.3f}")
print(f"{'Test Top-5 Accuracy':<25} | {f'{rnn_test_top5_acc:.2f}%':<15} | {f'{lstm_test_top5_acc:.2f}%':<15}")
print(f"{'Trainable Parameters':<25} | {'8,612,628':<15} | {'11,371,284':<15}")
print("\n**Conclusion:** The LSTM model performed significantly better than the simple RNN, achieving lower perplexity and higher accuracy. This is attributed to its ability to handle long-range dependencies in code more effectively, despite having more parameters and a longer training time.")


# --- 2. Results: Residual Connection ---
print("\n\n### 2. Results: Residual Connection (on Test Set)\n")
print(f"{'Metric':<25} | {'Baseline LSTM Model':<20} | {'Residual LSTM Model':<20}")
print("-"*70)
print(f"{'Test Perplexity (PPL)':<25} | {lstm_test_ppl:<20.3f} | {residual_lstm_test_ppl:<20.3f}")
print(f"{'Test Top-5 Accuracy':<25} | {f'{lstm_test_top5_acc:.2f}%':<20} | {f'{residual_lstm_test_top5_acc:.2f}%':<20}")

# --- Percentage Improvement Calculation ---
ppl_improvement = ((lstm_test_ppl - residual_lstm_test_ppl) / lstm_test_ppl) * 100
acc_improvement = residual_lstm_test_top5_acc - lstm_test_top5_acc

print("\n**Conclusion:** The addition of a residual connection provided a substantial performance boost. It improved both perplexity and accuracy without adding any extra trainable parameters, demonstrating that improving information flow within the network can lead to better learning.")
print("\n**Percentage Improvement:**")
print(f"- Perplexity saw a relative improvement of {ppl_improvement:.2f}%.")
print(f"- Top-5 Accuracy saw an absolute improvement of {acc_improvement:.2f}%.")
print("="*60)

              TASK 4: FINAL REPORT & SUBMISSION

### 1. Results: Baseline Comparison (on Test Set)

Metric                    | RNN Model       | LSTM Model     
------------------------------------------------------------
Test Perplexity (PPL)     | 7.206           | 5.515          
Test Top-5 Accuracy       | 80.73%          | 83.54%         
Trainable Parameters      | 8,612,628       | 11,371,284     

**Conclusion:** The LSTM model performed significantly better than the simple RNN, achieving lower perplexity and higher accuracy. This is attributed to its ability to handle long-range dependencies in code more effectively, despite having more parameters and a longer training time.


### 2. Results: Residual Connection (on Test Set)

Metric                    | Baseline LSTM Model  | Residual LSTM Model 
----------------------------------------------------------------------
Test Perplexity (PPL)     | 5.515                | 4.041               
Test Top-5 Accuracy       | 83.54%    